# Missão Aurora Siger — Ignition Zero
### Relatório de Pré-Decolagem: Sistemas, Verificação e Análise Operacional

---

| Campo | Informação |
|---|---|
| **Aluno** | Pedro de Oliveira Reis Sales |
| **RM** | 572709 |
| **Curso** | Bacharelado em Ciências da Computação |
| **Instituição** | FIAP |
| **Disciplina** | Atividade Integradora — Fase 1 |

---

> **Sobre este relatório:** Este documento descreve o sistema computacional embarcado responsável por validar as condições de pré-decolagem da nave Aurora Siger. Cada seção cobre uma camada do pipeline de verificação: coleta de dados, lógica de decisão, implementação em Python, balanço energético, análise de riscos assistida por IA e uma reflexão sobre as implicações dessa tecnologia no mundo real.

---
## Seção 1 — Telemetria de Pré-Decolagem

A telemetria é o mecanismo pelo qual a nave comunica seu estado interno e externo à equipe de controle em solo. Antes de qualquer decolagem, um conjunto de sensores embarcados registra variáveis críticas — temperatura, pressão, integridade estrutural, carga energética e status dos módulos principais.

Nesta simulação, esses dados são **gerados aleatoriamente dentro de intervalos realistas**, o que significa que cada execução pode produzir um cenário diferente: às vezes todos os sistemas estão operacionais, às vezes uma ou mais variáveis saem da faixa segura. Isso reflete o caráter imprevisível de operações espaciais reais.

### Parâmetros monitorados e suas faixas operacionais

| # | Parâmetro | Unidade | Faixa Operacional Segura | O que representa |
|---|---|---|---|---|
| 1 | Temperatura interna | °C | 18 a 28 | Ambiente interno habitável e seguro para os sistemas eletrônicos |
| 2 | Temperatura externa | °C | -115 a 15 | Condição atmosférica na zona de lançamento |
| 3 | Integridade estrutural | 0/1 | 1 (íntegra) | Ausência de danos na fuselagem detectados pelos sensores |
| 4 | Carga energética | % | ≥ 82% | Percentual mínimo de bateria para suportar todo o lançamento |
| 5 | Pressão dos propulsores | kPa | 310 a 390 | Pressão interna nos tanques de combustível |
| 6 | Módulo de propulsão | 0/1 | 1 (online) | Sistema de empuxo principal da nave |
| 7 | Módulo de comunicação | 0/1 | 1 (online) | Canal ativo com a base de controle em solo |
| 8 | Módulo de navegação | 0/1 | 1 (online) | Orientação, trajetória e posicionamento embarcado |

> **Nota técnica:** Os valores binários (0/1) representam estados discretos — ligado/desligado, íntegro/comprometido. Parâmetros contínuos (temperatura, pressão, energia) admitem um intervalo de valores aceitáveis; qualquer leitura fora desse intervalo aciona uma pendência no checklist.

In [ ]:
import random

# Para fixar os dados e ter sempre o mesmo resultado, ative a linha abaixo:
# random.seed(7)

def capturar_telemetria():
    """
    Simula a leitura dos sensores embarcados da nave Aurora Siger.

    Os intervalos foram calibrados para que falhas ocorram com
    probabilidade realista: módulos críticos têm 88% de chance de
    estarem online; parâmetros contínuos podem ultrapassar os limites.
    """
    return {
        "temp_interna":      round(random.uniform(14.0, 32.0), 1),
        "temp_externa":      round(random.uniform(-120.0, 22.0), 1),
        "integridade":       random.choices([1, 0], weights=[88, 12])[0],
        "carga_bateria":     round(random.uniform(68.0, 100.0), 1),
        "pressao_propulsor": round(random.uniform(285.0, 410.0), 1),
        "mod_propulsao":     random.choices([1, 0], weights=[88, 12])[0],
        "mod_comunicacao":   random.choices([1, 0], weights=[88, 12])[0],
        "mod_navegacao":     random.choices([1, 0], weights=[88, 12])[0],
    }


leitura = capturar_telemetria()

rotulos = {
    "temp_interna":      "Temperatura interna   (°C)",
    "temp_externa":      "Temperatura externa   (°C)",
    "integridade":       "Integridade estrutural [0/1]",
    "carga_bateria":     "Carga da bateria      (%)",
    "pressao_propulsor": "Pressão propulsores   (kPa)",
    "mod_propulsao":     "Módulo propulsão      [0/1]",
    "mod_comunicacao":   "Módulo comunicação    [0/1]",
    "mod_navegacao":     "Módulo navegação      [0/1]",
}

print()
print("  AURORA SIGER :: LEITURA DE SENSORES")
print("  " + "-" * 44)
for chave, rotulo in rotulos.items():
    print(f"  {rotulo:<32}  {leitura[chave]}")
print("  " + "-" * 44)
print()

---
## Seção 2 — Algoritmo de Verificação

Com os dados capturados pelos sensores, o próximo passo é aplicar um algoritmo de decisão que avalia cada parâmetro individualmente e emite um veredicto final sobre o lançamento.

A lógica é simples, mas robusta: **qualquer falha isolada é suficiente para suspender a missão**. Não existe compensação entre parâmetros — uma bateria excelente não cancela uma pressão fora do range.

### 2.1 Pseudocódigo

```
FUNÇÃO checar_condicoes(leitura):

    pendencias  ←  lista vazia

    SE leitura.temp_interna < 18 OU leitura.temp_interna > 28:
        registrar "Temperatura interna fora do intervalo seguro"

    SE leitura.temp_externa < -115 OU leitura.temp_externa > 15:
        registrar "Temperatura externa fora do intervalo seguro"

    SE leitura.integridade != 1:
        registrar "Comprometimento estrutural detectado"

    SE leitura.carga_bateria < 82:
        registrar "Carga energética insuficiente para decolagem"

    SE leitura.pressao_propulsor < 310 OU leitura.pressao_propulsor > 390:
        registrar "Pressão dos propulsores fora da faixa operacional"

    PARA CADA modulo EM [propulsao, comunicacao, navegacao]:
        SE modulo == 0:
            registrar "Módulo offline: " + nome

    SE pendencias VAZIA:
        RETORNAR "LANÇAMENTO AUTORIZADO"
    SENÃO:
        RETORNAR "LANÇAMENTO SUSPENSO" + pendencias

FIM
```

### 2.2 Estrutura lógica das verificações

O algoritmo percorre **8 verificações independentes**, agrupadas em 3 categorias:

| Categoria | Parâmetros verificados | Critério de aprovação |
|---|---|---|
| **Ambiente térmico** | Temperatura interna e externa | Valor dentro do intervalo numérico |
| **Hardware físico** | Integridade estrutural + pressão dos propulsores | Binário = 1 ou pressão no range |
| **Energia e módulos** | Carga da bateria + 3 módulos críticos | Bateria ≥ 82% e todos os módulos online |

Se todas as 8 verificações passarem → **LANÇAMENTO AUTORIZADO**  
Se ao menos uma falhar → **LANÇAMENTO SUSPENSO** com lista detalhada das pendências

---
## Seção 3 — Implementação em Python

A função abaixo traduz o pseudocódigo da seção anterior em Python funcional. Ela recebe o dicionário com os dados de telemetria, executa todas as verificações sequencialmente e imprime um relatório detalhado — incluindo qual parâmetro falhou e por quê.

In [ ]:
def checar_condicoes(dados):
    """
    Executa o checklist de pré-decolagem da nave Aurora Siger.

    Parâmetros avaliados:
      temp_interna      : faixa segura 18°C a 28°C
      temp_externa      : faixa segura -115°C a 15°C
      integridade       : deve ser 1 (estrutura íntegra)
      carga_bateria     : mínimo 82%
      pressao_propulsor : faixa segura 310 a 390 kPa
      mod_propulsao     : deve estar online (1)
      mod_comunicacao   : deve estar online (1)
      mod_navegacao     : deve estar online (1)

    Retorna True se autorizado, False se suspenso.
    """
    pendencias = []

    # --- Ambiente térmico ---
    if not (18 <= dados["temp_interna"] <= 28):
        pendencias.append(
            f"Temp. interna fora do intervalo: {dados['temp_interna']}°C  [seguro: 18-28°C]"
        )

    if not (-115 <= dados["temp_externa"] <= 15):
        pendencias.append(
            f"Temp. externa fora do intervalo: {dados['temp_externa']}°C  [seguro: -115 a 15°C]"
        )

    # --- Integridade física ---
    if dados["integridade"] != 1:
        pendencias.append("Comprometimento estrutural detectado pelos sensores de fuselagem")

    # --- Energia ---
    if dados["carga_bateria"] < 82:
        pendencias.append(
            f"Carga insuficiente: {dados['carga_bateria']}%  [mínimo exigido: 82%]"
        )

    # --- Propulsão ---
    if not (310 <= dados["pressao_propulsor"] <= 390):
        pendencias.append(
            f"Pressão dos propulsores fora da faixa: {dados['pressao_propulsor']} kPa  [seguro: 310-390 kPa]"
        )

    # --- Módulos críticos ---
    modulos = {
        "Propulsão":   dados["mod_propulsao"],
        "Comunicação": dados["mod_comunicacao"],
        "Navegação":   dados["mod_navegacao"],
    }
    for nome, status in modulos.items():
        if status != 1:
            pendencias.append(f"Módulo de {nome} reportado como offline")

    # --- Veredicto ---
    print()
    print("  AURORA SIGER :: CHECKLIST DE PRÉ-DECOLAGEM")
    print("  " + "=" * 50)

    if not pendencias:
        print("  VEREDICTO  >>  LANÇAMENTO AUTORIZADO")
        print("  Todos os sistemas operacionais. Go for launch.")
    else:
        print("  VEREDICTO  >>  LANÇAMENTO SUSPENSO")
        print(f"  {len(pendencias)} pendência(s) registrada(s):")
        for item in pendencias:
            print(f"    [!] {item}")

    print("  " + "=" * 50)
    print()
    return len(pendencias) == 0


autorizado = checar_condicoes(leitura)

### 3.1 Simulação de cenário crítico

Para validar que o algoritmo rejeita corretamente entradas fora dos limites, simulamos abaixo um cenário com **quatro falhas simultâneas**: temperatura interna elevada, bateria abaixo do limite, pressão excessiva nos propulsores e módulo de navegação offline.

In [ ]:
# Cenário de stress test — falhas deliberadamente injetadas para validar o algoritmo
cenario_critico = {
    "temp_interna":      31.4,  # [!] acima de 28°C
    "temp_externa":     -88.0,  # OK
    "integridade":           1, # OK
    "carga_bateria":     78.2,  # [!] abaixo de 82%
    "pressao_propulsor": 402.0, # [!] acima de 390 kPa
    "mod_propulsao":         1, # OK
    "mod_comunicacao":       1, # OK
    "mod_navegacao":         0, # [!] offline
}

print("  --- ENTRADA: CENÁRIO CRÍTICO (DADOS FORÇADOS) ---")
for k, v in cenario_critico.items():
    print(f"  {k:<22} : {v}")

checar_condicoes(cenario_critico)

---
## Seção 4 — Balanço Energético

A análise energética vai além de checar se a bateria está acima do limite mínimo. Ela quantifica **quanto da energia disponível é efetivamente aproveitada**, desconta as perdas inerentes ao sistema e calcula quanto sobra após o consumo do lançamento — valor que determina a autonomia real da missão.

O modelo adotado considera:

| Componente | Descrição |
|---|---|
| Capacidade instalada | Total de kWh das baterias da nave |
| Energia utilizável | Capacidade × percentual de carga atual |
| Perdas de conversão | 6% — resistência elétrica, calor dissipado e distribuição |
| Consumo do lançamento | Energia gasta do ignition até estabilização orbital |
| Reserva operacional | O que sobra para sustentar os sistemas durante a missão |

In [ ]:
# Especificações energéticas da nave Aurora Siger
CAPACIDADE_TOTAL_KWH = 520   # kWh — capacidade máxima das baterias
PERDA_CONVERSAO_PCT  = 6     # % — perdas estimadas por calor e resistência
CONSUMO_LANCAMENTO   = 140   # kWh — gasto do ignition até órbita estável
CONSUMO_CRUZEIRO     = 18    # kWh/h — consumo médio em voo de cruzeiro

# Variável dinâmica: vem da telemetria capturada
carga_pct = leitura["carga_bateria"]

# Cálculos
energia_bruta   = CAPACIDADE_TOTAL_KWH * (carga_pct / 100)
perdas_kwh      = energia_bruta * (PERDA_CONVERSAO_PCT / 100)
energia_liquida = energia_bruta - perdas_kwh
reserva_missao  = energia_liquida - CONSUMO_LANCAMENTO
autonomia_h     = max(reserva_missao / CONSUMO_CRUZEIRO, 0)

# Relatório
print()
print("  AURORA SIGER :: BALANÇO ENERGÉTICO")
print("  " + "-" * 46)
print(f"  Capacidade instalada            : {CAPACIDADE_TOTAL_KWH} kWh")
print(f"  Carga atual (telemetria)         : {carga_pct}%")
print(f"  Energia bruta disponível         : {energia_bruta:.1f} kWh")
print(f"  Perdas de conversão ({PERDA_CONVERSAO_PCT}%)       : -{perdas_kwh:.1f} kWh")
print(f"  Energia líquida aproveitável     : {energia_liquida:.1f} kWh")
print(f"  Consumo estimado do lançamento   : -{CONSUMO_LANCAMENTO} kWh")
print("  " + "-" * 46)
print(f"  Reserva para a missão            : {reserva_missao:.1f} kWh")
print(f"  Autonomia estimada em cruzeiro   : {autonomia_h:.1f} horas")
print("  " + "-" * 46)

if reserva_missao < 0:
    print("  [CRITICO] Energia insuficiente para completar o lancamento!")
elif reserva_missao < 80:
    print("  [ATENCAO] Reserva abaixo de 80 kWh — margem operacional reduzida.")
else:
    print("  [OK] Reserva dentro dos parametros. Missao energeticamente viavel.")
print()

---
## Seção 5 — Análise de Riscos Assistida por IA

Além do checklist binário (passa/não passa), uma análise de risco mais refinada classifica cada parâmetro em **níveis de criticidade** — não apenas se está dentro ou fora do limite, mas o quão próximo está da borda crítica. Isso permite antecipar problemas antes que se tornem falhas formais.

Esta análise foi desenvolvida com apoio de IA (Claude — Anthropic), que sugeriu os critérios de gradação de risco e identificou padrões de vulnerabilidade relevantes para missões de curta duração.

### 5.1 Escala de criticidade adotada

| Nível | Indicador | Critério |
|---|---|---|
| NOMINAL | 🟢 | Dentro da faixa segura com margem confortável |
| ATENCAO | 🟡 | Dentro da faixa, mas próximo do limite (menos de 10% de margem) |
| CRITICO | 🔴 | Fora da faixa operacional — pendência registrada no checklist |

### 5.2 Mapa de riscos identificados pela IA

| Risco | Probabilidade | Impacto | Mitigação recomendada |
|---|---|---|---|
| Energia próxima ao limiar mínimo | Média | Alto | Threshold de alerta interno em 87% (5pp de antecipação) |
| Módulo offline sem redundância | Baixa | Crítico | Módulos backup para propulsão e navegação |
| Pressão dos propulsores oscilando | Baixa | Alto | Monitoramento contínuo com alertas a 325 e 375 kPa |
| Temperatura interna em elevação | Baixa | Médio | Ventilação ativa acionada acima de 25°C |
| Temperatura externa extrema | Média | Médio | Verificação de isolamento térmico antes do lançamento |

In [ ]:
def classificar_risco(dados):
    """
    Análise de risco graduada para cada parâmetro de telemetria.
    Vai além do binário: avalia a proximidade dos limites e aponta tendências.
    """
    relatorio = []

    # Temperatura interna
    ti = dados["temp_interna"]
    if 18 <= ti <= 28:
        nivel = "🟡 ATENCAO" if ti > 26 else "🟢 NOMINAL"
        obs   = f"{ti}°C" + (" — próximo do limite superior (28°C)" if ti > 26 else "")
    else:
        nivel = "🔴 CRITICO"
        obs   = f"{ti}°C — FORA DA FAIXA SEGURA (18-28°C)"
    relatorio.append(("Temp. interna", nivel, obs))

    # Temperatura externa
    te = dados["temp_externa"]
    if -115 <= te <= 15:
        nivel = "🟡 ATENCAO" if te < -100 else "🟢 NOMINAL"
        obs   = f"{te}°C" + (" — extrema, verificar isolamento externo" if te < -100 else "")
    else:
        nivel = "🔴 CRITICO"
        obs   = f"{te}°C — FORA DA FAIXA SEGURA (-115 a 15°C)"
    relatorio.append(("Temp. externa", nivel, obs))

    # Integridade
    ig = dados["integridade"]
    relatorio.append((
        "Integridade",
        "🟢 NOMINAL" if ig == 1 else "🔴 CRITICO",
        "Estrutura integra" if ig == 1 else "DANO ESTRUTURAL DETECTADO"
    ))

    # Carga da bateria
    cb = dados["carga_bateria"]
    if cb >= 82:
        nivel = "🟡 ATENCAO" if cb < 90 else "🟢 NOMINAL"
        obs   = f"{cb}%" + (f" — margem de {cb-82:.1f}pp acima do minimo" if cb < 90 else "")
    else:
        nivel = "🔴 CRITICO"
        obs   = f"{cb}% — ABAIXO DO MINIMO DE 82%"
    relatorio.append(("Carga bateria", nivel, obs))

    # Pressão propulsores
    pp = dados["pressao_propulsor"]
    if 310 <= pp <= 390:
        nivel = "🟡 ATENCAO" if pp < 325 or pp > 375 else "🟢 NOMINAL"
        obs   = f"{pp} kPa"
        if pp < 325:  obs += " — proximo do limite inferior (310 kPa)"
        elif pp > 375: obs += " — proximo do limite superior (390 kPa)"
    else:
        nivel = "🔴 CRITICO"
        obs   = f"{pp} kPa — FORA DA FAIXA (310-390 kPa)"
    relatorio.append(("Pressao propulsor", nivel, obs))

    # Módulos
    for nome_exib, chave in [("Mod. Propulsao",   "mod_propulsao"),
                              ("Mod. Comunicacao", "mod_comunicacao"),
                              ("Mod. Navegacao",   "mod_navegacao")]:
        st = dados[chave]
        relatorio.append((
            nome_exib,
            "🟢 NOMINAL" if st == 1 else "🔴 CRITICO",
            "Online" if st == 1 else "OFFLINE — sistema inoperante"
        ))

    # Exibição
    print()
    print("  AURORA SIGER :: ANALISE DE RISCO (IA)")
    print(f"  {'Parametro':<22} {'Status':<18} Observacao")
    print("  " + "-" * 70)
    for param, status, obs in relatorio:
        print(f"  {param:<22} {status:<18} {obs}")
    print("  " + "-" * 70)

    criticos = sum(1 for _, s, _ in relatorio if "CRITICO" in s)
    atencao  = sum(1 for _, s, _ in relatorio if "ATENCAO" in s)
    nominais = sum(1 for _, s, _ in relatorio if "NOMINAL" in s)
    print(f"  Resumo: {nominais} nominal(is)  |  {atencao} atencao  |  {criticos} critico(s)")
    print()


classificar_risco(leitura)

---
## Seção 6 — Reflexão Crítica

### 6.1 Ética e Responsabilidade na Engenharia de Sistemas Críticos

Um sistema de verificação de pré-decolagem não é apenas código — é uma cadeia de decisões com consequências diretas sobre vidas humanas e patrimônio. Do ponto de vista da Ciência da Computação, isso coloca uma questão ética central na mesa: **qual é o grau de responsabilidade do desenvolvedor quando um algoritmo decide se uma pessoa pode ser lançada ao espaço?**

A resposta começa antes do código: nas premissas. Quem define que 82% é o limite mínimo de bateria? Quem valida que 310–390 kPa é seguro para esses propulsores? Erros nessas definições não são bugs de programação — são falhas de engenharia com potencial catastrófico. A responsabilidade ética do desenvolvedor inclui questionar os parâmetros recebidos, documentá-los com clareza e garantir que o sistema não tome decisões além do seu escopo de competência.

Auditabilidade também importa. Um sistema que apenas imprime "AUTORIZADO" sem explicar o raciocínio é opaco por design. O algoritmo desta atividade já aponta na direção correta: ao listar cada pendência individualmente, ele permite que um operador humano avalie e, se necessário, sobrescreva a decisão com responsabilidade consciente — não de forma cega.

### 6.2 Impacto Social da Exploração Espacial

A corrida espacial do século XXI é diferente da do século XX. Governos ainda participam, mas empresas privadas mudaram o ritmo, o custo e — criticamente — a narrativa. O espaço deixou de ser um projeto coletivo da humanidade para se tornar, em partes crescentes, um mercado com seus próprios desequilíbrios.

Os custos de lançamento caíram de forma expressiva na última década. Isso democratizou o acesso a certas tecnologias espaciais — satélites de monitoramento climático, sistemas de alerta de desastres, comunicação em regiões remotas. Mas o acesso à infraestrutura de lançamento ainda é assimétrico: nações sem capacidade própria dependem de parcerias cujas condições raramente são igualitárias.

O desafio real está em garantir que as inovações geradas pela exploração espacial sejam tratadas como bem público — não apenas como vantagem competitiva privada. Tecnologia desenvolvida com investimento público (incluindo pesquisa acadêmica) deveria retornar à sociedade de forma acessível. Esse princípio é tão válido para satélites quanto para qualquer software desenvolvido em universidades.

### 6.3 Sustentabilidade Tecnológica e o Problema do Lixo Orbital

Há hoje mais de 27.000 fragmentos de detritos orbitais rastreáveis acima de 10 cm, e estima-se que existam milhões de fragmentos menores, indetectáveis pelos sistemas atuais. Cada um deles orbita a cerca de 7,5 km/s — velocidade suficiente para que um objeto do tamanho de uma ervilha cause danos estruturais sérios a uma nave.

A sustentabilidade espacial não é tema do futuro — é problema do presente. O chamado síndrome de Kessler descreve um cenário em que a densidade de detritos em certas órbitas se torna grande o suficiente para gerar colisões em cascata, tornando aquelas faixas inutilizáveis por décadas. Projetos responsáveis precisam incluir planos de deorbitação, propulsão eficiente e materiais que se fragmentam de forma controlada na reentrada.

Do ponto de vista computacional, sustentabilidade também aparece na eficiência dos sistemas embarcados: algoritmos que consomem menos energia no processamento, que falham de forma controlada sem travar a nave, que geram logs compactos e auditáveis. Isso não é detalhe de implementação — é princípio de design responsável.

> **Síntese:** A tecnologia espacial é um espelho ampliado das escolhas que fazemos na Terra. Quem tem acesso, quem arca com os riscos, quem herda os resíduos — essas perguntas não têm respostas técnicas. Têm respostas éticas, políticas e sociais. O papel do cientista da computação nesse cenário é construir sistemas transparentes o suficiente para que essas perguntas possam ser feitas — e respondidas por quem de direito.